# Cookbook: Bulk Data Pipeline

Build a data pipeline that processes multiple companies at scale.
This notebook demonstrates batch processing, SEC dataset access,
rate limiting, and exporting aggregated results.

**What you'll learn:**
1. Process a portfolio of companies in a loop
2. Aggregate financial metrics across companies
3. Access SEC public datasets and taxonomies
4. Respect SEC rate limits with built-in throttling
5. Export pipeline results to DataFrames and CSV

## 1. Setup

In [ ]:
from edgar.client import EdgarClient

# The client enforces SEC's 10 req/sec rate limit automatically.
# You can lower it for conservative pipelines.
edgar_client = EdgarClient(
    user_agent="Your Name your-email@example.com",
    rate_limit=8,   # Stay under SEC's 10 req/sec limit
    cache=True,      # Cache responses to avoid redundant requests
)

## 2. Define a Company Universe

Start with a list of tickers to process. The library resolves each to a CIK.

In [2]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]

companies = {}
for ticker in tickers:
    company = edgar_client.company(ticker)
    companies[ticker] = company
    print(f"{ticker:<6} → CIK: {company.cik}  {company.name}")

AAPL   → CIK: 0000320193  Apple Inc.
MSFT   → CIK: 0000789019  MICROSOFT CORP
GOOGL  → CIK: 0001652044  Alphabet Inc.
AMZN   → CIK: 0001018724  AMAZON COM INC
META   → CIK: 0001326801  Meta Platforms, Inc.


## 3. Batch Retrieve Company Metadata

Pull metadata for every company in the universe. The TTL cache ensures
repeated lookups within the session don't hit the SEC API again.

In [3]:
metadata = {}
for ticker, company in companies.items():
    info = company.get_info()
    if info:
        metadata[ticker] = {
            "name": info.name,
            "cik": info.cik,
            "sic": info.sic,
            "sic_description": info.sic_description,
            "fiscal_year_end": info.fiscal_year_end,
            "recent_filing_count": len(info.recent_submissions),
        }

print(f"{'Ticker':<8} {'SIC':<6} {'FY End':<8} {'Recent Filings':>15}")
print("-" * 45)
for ticker, m in metadata.items():
    print(f"{ticker:<8} {m['sic']:<6} {m['fiscal_year_end']:<8} {m['recent_filing_count']:>15}")

Ticker   SIC    FY End    Recent Filings
---------------------------------------------
AAPL     3571   0926                1002
MSFT     7372   0630                1009
GOOGL    7370   1231                1023
AMZN     5961   1231                1000
META     7370   1231                1003


## 4. Aggregate a Financial Metric Across Companies

Pull Revenue for each company and build a comparison table.

In [4]:
revenue_comparison = []

for ticker, company in companies.items():
    facts = company.get_facts()
    if not facts:
        continue

    revenue = facts.get("us-gaap", "Revenues", unit="USD")
    # Keep only annual (10-K) data.
    annual = [r for r in revenue if r.form == "10-K"]

    for r in annual[-3:]:  # Last 3 years
        revenue_comparison.append({
            "ticker": ticker,
            "company": company.name,
            "fiscal_year": r.fiscal_year,
            "period": r.fiscal_period,
            "revenue": r.value,
            "filed": r.filed,
        })

print(f"Collected {len(revenue_comparison)} data points across {len(companies)} companies.")

Collected 12 data points across 5 companies.


In [5]:
# Display as a DataFrame.
from edgar.models import to_dataframe
import pandas as pd

revenue_df = pd.DataFrame(revenue_comparison)
revenue_df["revenue_billions"] = revenue_df["revenue"].astype(float) / 1e9
revenue_df[["ticker", "fiscal_year", "revenue_billions", "filed"]].sort_values(
    ["fiscal_year", "revenue_billions"], ascending=[False, False]
)

,ticker,fiscal_year,revenue_billions,filed
8,GOOGL,2025,402.836,2026-02-05
7,GOOGL,2025,350.018,2026-02-05
6,GOOGL,2025,307.394,2026-02-05
1,AAPL,2018,265.595,2018-11-05
2,AAPL,2018,62.900,2018-11-05
0,AAPL,2018,53.265,2018-11-05
11,META,2017,40.653,2018-02-01
10,META,2017,27.638,2018-02-01
9,META,2016,27.638,2017-02-03
4,MSFT,2010,62.484,2010-07-30


## 5. Pivot Table — Revenue by Company and Year

In [6]:
pivot = revenue_df.pivot_table(
    index="ticker",
    columns="fiscal_year",
    values="revenue_billions",
    aggfunc="first",
)
pivot.round(1)

fiscal_year,2010,2016,2017,2018,2025
ticker,,,,,
AAPL,NaN,NaN,NaN,53.3,NaN
GOOGL,NaN,NaN,NaN,NaN,307.4
META,NaN,27.6,27.6,NaN,NaN
MSFT,14.5,NaN,NaN,NaN,NaN


## 6. Batch Filing Count by Form Type

Count recent filings by type for each company.

In [7]:
filing_counts = []

for ticker, company in companies.items():
    filings = company.get_filings()
    form_counts = {}
    for f in filings:
        form_counts[f.form_type] = form_counts.get(f.form_type, 0) + 1

    for form_type, count in form_counts.items():
        filing_counts.append({
            "ticker": ticker,
            "form_type": form_type,
            "count": count,
        })

counts_df = pd.DataFrame(filing_counts)
pivot_filings = counts_df.pivot_table(
    index="form_type",
    columns="ticker",
    values="count",
    aggfunc="sum",
    fill_value=0,
)

# Show the most common form types.
pivot_filings.loc[pivot_filings.sum(axis=1).nlargest(10).index]

ticker,AAPL,AMZN,GOOGL,META,MSFT
form_type,,,,,
4,62,59,62,65,85
144,18,26,28,41,4
8-K,13,9,5,5,4
PX14A6G,3,0,0,1,12
SCHEDULE 13G/A,2,2,3,1,1
10-Q,4,2,0,0,2
3,4,0,1,2,1
424B5,0,6,2,0,0
5,0,0,7,0,0


## 7. SEC Public Datasets

The SEC publishes bulk datasets including all XBRL taxonomies.
Use the Datasets service to discover what's available.

In [9]:
datasets_service = edgar_client.datasets()
taxonomies = datasets_service.get_edgar_taxonomies()

print(f"EDGAR taxonomies available: {len(taxonomies)}\n")
for t in taxonomies[:10]:
    print(f"  {t}")

TypeError: a bytes-like object is required, not 'NoneType'

## 8. Export Pipeline Results

Save aggregated data to CSV for downstream analysis.

In [ ]:
# Uncomment to save to CSV:
# revenue_df.to_csv("revenue_comparison.csv", index=False)
# pivot.to_csv("revenue_pivot.csv")
# print("Exported to CSV files.")

# For this demo, show the export approach:
print("Export commands:")
print('  revenue_df.to_csv("revenue_comparison.csv", index=False)')
print('  pivot.to_csv("revenue_pivot.csv")')
print(f"\nDataFrame shape: {revenue_df.shape}")
print(f"Pivot shape:     {pivot.shape}")

## 9. Rate Limiting & Caching in Action

The client automatically throttles requests and caches responses.
Here's how to verify.

In [ ]:
# The cache avoids redundant API calls.
# Fetching Apple's facts again will hit the cache, not SEC.
import time

start = time.time()
facts_cached = edgar_client.company("AAPL").get_facts()
elapsed = time.time() - start

print(f"Cached XBRL fetch: {elapsed:.4f}s")
print(f"Entity: {facts_cached.entity_name}")
print("\nTip: Set cache=False for real-time data in production pipelines:")
print('  client = EdgarClient(user_agent="...", cache=False)')

## Summary

| Task | Method | Notes |
|------|--------|-------|
| Configure rate limit | `EdgarClient(rate_limit=8)` | Default 10 req/sec (SEC max) |
| Enable/disable cache | `EdgarClient(cache=True)` | 24h TTL for tickers/taxonomy, 1h for submissions |
| Resolve tickers | `client.company("AAPL")` | Cached after first fetch |
| Batch metadata | `company.get_info()` | Returns `CompanyInfo` model |
| Batch XBRL facts | `company.get_facts()` | Returns `Facts` model |
| Aggregate metrics | `facts.get("us-gaap", "Revenues")` | Filter by unit, period |
| SEC datasets | `client.datasets().get_edgar_taxonomies()` | Bulk reference data |
| Export | `df.to_csv("output.csv")` | Standard pandas export |

**Rate limit best practices:**
- Use `rate_limit=8` for batch pipelines (leaves headroom)
- Keep `cache=True` to avoid redundant requests
- Process companies sequentially — the throttle handles timing automatically